# 🎙️🎬 NỖI SỢ AUDIOBOOK - STUDIO V20.0 (CINEMA GOTHIC V20)
### 🦇 Động Cơ Subtitle-Locked Scene Sync & All-in-One Gemini 3.8 Flash Director

Hệ thống tự động hóa sản xuất Video Essay & Audiobook Trinh Thám - Kinh Dị Gothic chất lượng cao:
- 📺 **Hồ Sơ Đa Kênh 1-Click (7 Profiles Preset)**: Nỗi Sợ AudioBook (Gothic Noir), Phật Pháp, Lịch Sử, Khoa Học Viễn Tưởng, Podcast Cuộc Sống, Cổ Tích, Custom Profile.
- ⚡ **Khóa Chặt Phân Cảnh Theo Subtitle (Subtitle-Locked Scene Sync)**: 100% hình đi cùng sub, bắt đầu câu thoại nào ảnh xuất hiện ngay lập tức, độ dài lý tưởng 2.4s - 6.8s.
- 🤖 **AI Đạo Diễn All-in-One (Gemini 3.8 / 3.5 Flash Lite)**: 1 API Call duy nhất cho cả kịch bản hình ảnh và YouTube Chapters SEO, tiết kiệm quota 90%.
- 🎬 **Kho Video Stock Đa Nguồn (Pexels / Coverr / Mixkit)**: Tự động xoay tua API key và chống trùng lặp cảnh.
- 🖼️ **Thumbnail CTR Booster (1280x720 HD)**: Tự động trích xuất khung hình cao trào từ video và gán typography nổi bật.
- 🚀 **Tự Động Đăng & Quản Lý Xuất Bản YouTube**: Đăng tải đa kênh với token riêng biệt và ghi nhận tiến độ vào Google Drive.


In [11]:
# @title ⚡ 0. Kiểm Tra & Kích Hoạt Mặc Định GPU T4
import torch
import sys

if not torch.cuda.is_available():
    print("❌ BẠN ĐANG CHẠY TRÊN CPU! Hãy chuyển sang GPU T4 để tăng tốc xử lý:")
    print("👉 Vào Menu trên Colab: Runtime (Thời gian chạy) -> Change runtime type (Thay đổi loại thời gian chạy) -> Chọn T4 GPU -> Bấm Save (Lưu).")
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🚀 GPU đã kích hoạt: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    if "T4" in gpu_name:
        print("✅ Chuẩn cấu hình: Đã kết nối GPU Tesla T4 tối ưu cho VieNeu-TTS & SDXL!")
    else:
        print(f"⚡ Đang chạy trên GPU hiệu năng cao: {gpu_name}")


🚀 GPU đã kích hoạt: Tesla T4 (14.6 GB VRAM)
✅ Chuẩn cấu hình: Đã kết nối GPU Tesla T4 tối ưu cho VieNeu-TTS & SDXL!


In [12]:
# @title 1. Kết Nối Google Drive (Tùy chọn) & Tải Mã Nguồn Từ GitHub
import os
from google.colab import drive
try:
    drive.mount("/content/drive")
    print("✅ Đã kết nối Google Drive!")
except Exception as e:
    print(f"ℹ️ Bỏ qua mount Drive: {e}")

# Tải hoặc cập nhật repository
%cd /content
if not os.path.exists("audiobook-automation"):
    !git clone https://github.com/tuna1710/audiobook-automation.git
    %cd /content/audiobook-automation
else:
    %cd /content/audiobook-automation
    !git pull


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã kết nối Google Drive!
/content
/content/audiobook-automation
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 549 bytes | 549.00 KiB/s, done.
From https://github.com/tuna1710/audiobook-automation
   1581437..f98a571  main       -> origin/main
Updating 1581437..f98a571
Fast-forward
 src/youtube/__init__.py | 4 +++-
 1 file changed, 3 insertions(+), 1 deletion(-)


In [13]:
# @title 2. Cài Đặt Thư Viện Cần Thiết
%cd /content/audiobook-automation
!pip install -q vieneu diffusers accelerate transformers gradio pillow requests openai-whisper gdown google-api-python-client google-auth-oauthlib google-auth-httplib2 google-generativeai pyyaml python-dotenv soundfile
print("✅ Đã cài đặt hoàn tất toàn bộ thư viện!")


/content/audiobook-automation
✅ Đã cài đặt hoàn tất toàn bộ thư viện!


In [14]:
# @title 2.5 (Tùy Chọn) Đồng Bộ File OAuth YouTube Từ Google Drive
import os
import shutil

# Nếu bạn đã lưu sẵn client_secret.json hoặc token.json trên Google Drive, tự động copy vào thư mục làm việc
drive_folder = "/content/drive/MyDrive"
for filename in ["client_secret.json", "client_secrets.json", "token.json", "youtube_token.json"]:
    drive_file = os.path.join(drive_folder, filename)
    if os.path.exists(drive_file):
        shutil.copy2(drive_file, f"/content/audiobook-automation/{filename}")
        shutil.copy2(drive_file, f"/content/{filename}")
        print(f"✅ Đã tự động đồng bộ {filename} từ Google Drive!")

print("ℹ️ Nếu chưa có file client_secret.json, bạn có thể tải trực tiếp tại TAB 3 trên giao diện Web Studio!")


ℹ️ Nếu chưa có file client_secret.json, bạn có thể tải trực tiếp tại TAB 3 trên giao diện Web Studio!


In [ ]:
# @title 3. Khởi Chạy Web Studio (Nhận Link Gradio Share)
%cd /content/audiobook-automation
import os

# Tự động nạp Gemini API Key từ Colab Secrets (Biểu tượng chìa khóa 🔑)
try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
    if gemini_key:
        os.environ['GEMINI_API_KEY'] = str(gemini_key).strip()
        print('🔑 Đã tự động kết nối GEMINI_API_KEY từ Colab Secrets (Chìa khóa 🔑)!')
except Exception:
    pass

!python main.py --share


/content/audiobook-automation
🔑 Đã tự động kết nối GEMINI_API_KEY từ Colab Secrets (Chìa khóa 🔑)!
🎙️ AUDIOBOOK AUTOMATION STUDIO V20.0 (Cinema Gothic V20)
🌐 Server: http://localhost:7860
🔗 Chế độ chia sẻ trực tuyến (--share): BẬT (Thích hợp cho Google Colab)
🚀 PHẦN CỨNG: GPU Tesla T4 (14.6 GB VRAM) - FP16 Kích Hoạt Tối Ưu
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://891aa6835d5993002b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/grad